In [ ]:
import marimo as mo
import pandas as pd
import numpy as np
import altair as alt

In [ ]:
# Import the Titanic dataset
df = pd.read_csv("data/titanic.csv")
df["Cabin"] = df["Cabin"].str[0]
df.sample(5)

### **Dataset variables:**

*   survival -->	Whether a passenger survived or not
*   pclass -->	Ticket class (1 = 1st, 2 = 2nd, 3 = 3rd)
*   sex -->	Sex
*   Age -->	Age in years
*   sibsp -->	# of siblings / spouses aboard the Titanic
*   parch -->	# of parents / children aboard the Titanic
*   ticket -->	Ticket number
*   fare -->	Passenger fare
*   cabin -->	Cabin number
*   embarked -->	Port of Embarkation (C = Cherbourg, Q = Queenstown, S = Southampton)

In [ ]:
## Dropping irrelevant columns and target variable

X = df.drop(columns=["PassengerId", "Name", "Ticket"])
y = X.pop("Survived")

##checking null values of predictors
X.isna().sum()

### **Splitting data**

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## Dividing the training set into categorical and numeric
X_train_num = X_train.select_dtypes(include="number")
X_train_cat = X_train.select_dtypes(exclude="number")

## Dividing the test set into categorical and numeric
X_test_cat = X_test.select_dtypes(include='object')
X_test_num = X_test.select_dtypes(include='number')

### **Creating null model or baseline**

In [ ]:
from sklearn.metrics import accuracy_score

## creating null model
pred_pessimistic_train = pd.Series(0, index=y_train.index)
pred_pessimistic_test = pd.Series(0, index=y_test.index)

acc_train_null = accuracy_score(y_true = y_train,
                                 y_pred = pred_pessimistic_train)

acc_test_null = accuracy_score(y_true = y_test,
                                 y_pred = pred_pessimistic_test
                                 )

acc_train_null, acc_test_null

## **Numerical Pipeline**

### **Initializing basic pipeline only for numerical**

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn import set_config

imputer = SimpleImputer(strategy="median")
scaler = StandardScaler()
dtree = DecisionTreeClassifier(max_depth=4,
                               min_samples_leaf=10,
                               random_state=42)

### **Creating Pipeline**

In [ ]:
pipe = make_pipeline(imputer,scaler, dtree).set_output(transform='pandas')
pipe

### **Fitting Pipeline**

In [ ]:
pipe.fit(X_train_num, y_train)

### **Making predictions**

In [ ]:
train_pred = pipe.predict(X_train_num)
test_pred = pipe.predict(X_test_num)

### **Measuring acccuracy**

In [ ]:
acc_train_num = accuracy_score(y_train, train_pred)
acc_test_num = accuracy_score(y_test, test_pred)

acc_train_num, acc_test_num

## **Numerical and categorical Pipeline**

### **Imputing missing values for categorical variables**

Before implementing a pipeline, we are going to encode the categorical features manually

In [ ]:
## Have a look at the training categorical set
X_train_cat.describe()

In [ ]:
## Imputing missing values
# defining the imputer to use "unknown" as replacement value
cat_imputer = SimpleImputer(strategy="constant",
                            fill_value="unknown").set_output(transform='pandas')

# fitting and transforming
X_cat_imputed = cat_imputer.fit_transform(X_train_cat)

### **Learning One Hot Encoding**

Because machines cannot understand categories or language until they are represented numerically, we have encode or convert into a categorical format the categorical features

In [ ]:
from sklearn.preprocessing import OneHotEncoder

# initializing the Encoder
onehot = OneHotEncoder(drop="first",sparse_output=False).set_output(transform='pandas')

# fiting 
onehot.fit(X_cat_imputed)

# transforming
X_cat_imputed_onehot = onehot.transform(X_cat_imputed)

In [ ]:
X_cat_imputed_onehot

### **Creating Pipeline including encoder**

In [ ]:
from sklearn.compose import make_column_transformer

# create numerical pipeline, only with the scaler and SimpleImputer(strategy="mean")
numeric_pipe = make_pipeline(
    SimpleImputer(strategy="mean"),
    StandardScaler()
)

 # create categorical pipeline, with the SimpleImputer(fill_value="missing") and the OneHotEncoder
categoric_pipe = make_pipeline(
    SimpleImputer(strategy="constant", fill_value="missing"),
    OneHotEncoder(sparse_output=False, handle_unknown="ignore", drop="first")
)

preprocessor = make_column_transformer(
    (numeric_pipe, X_train_num.columns),
    (categoric_pipe, X_train_cat.columns),
)

preprocessor

In [ ]:
## Putting everything together
cat_num_pipeline = make_pipeline(preprocessor,dtree)

## Fitting Pipeline
cat_num_pipeline.fit(X_train, y_train)

## Predicting
cat_num_train_predicted = cat_num_pipeline.predict(X_train)

In [ ]:
acc_train_catnum = accuracy_score(y_train,cat_num_train_predicted)

acc_train_null, acc_train_num, acc_train_catnum,

In [ ]:
cat_num_test_predicted = cat_num_pipeline.predict(X_test)
acc_test_catnum = accuracy_score(y_test,cat_num_test_predicted)

acc_test_null, acc_test_num, acc_test_catnum

## Optimizing model parameters through GridSearch and Cross Validation

The problem is that we don't know what are the best model parameters that work best. We can run the thing over and over with different parameters but it is not efficient. The only difference is that we include our pipeline into a search grid. The search grid, based on the range of values for the parameters we want to optimize, will find the best parameters.

The key component is the K-Fold given by the parameter cv:
*It gives you a more reliable estimate of model performance compared to a single train-test split. Every data point gets used for both training and testing (just not at the same time), which reduces the risk that your performance estimate is biased by one particular split of the data.*

Imagine you set **cv = 5**. The process goes as follows:

Your entire dataset is randomly divided into 5 equal parts (folds)
The model is trained and evaluated 5 times, each time using a different fold as the test set:

Iteration 1: Train on folds 2,3,4,5 → Test on fold 1<br>
Iteration 2: Train on folds 1,3,4,5 → Test on fold 2<br>
Iteration 3: Train on folds 1,2,4,5 → Test on fold 3<br>
Iteration 4: Train on folds 1,2,3,5 → Test on fold 4<br>
Iteration 5: Train on folds 1,2,3,4 → Test on fold 5

Each iteration produces an accuracy score, so you get 5 accuracy scores total
GridSearchCV averages these 5 scores to get the final performance metric for each parameter combination

For this part, we will:
1. Reuse the <code>preprocessor</code>
2. Initialize another <code>DecisionTreeClassifier()</code> but without parameters
3. Create another  <code>pipeline</code>
4. Set parameters for GridSearch
5. Implement the <code>GridSearch()</code>

In [ ]:
from sklearn.model_selection import GridSearchCV

## Initializing new model tree
new_dtree = DecisionTreeClassifier()

## Creating Pipeline with new_dtree
grid_pipeline = make_pipeline(preprocessor,new_dtree).set_output(transform='pandas')

## setting the parameters
grid_parameters = {
    'decisiontreeclassifier__max_depth': range(2, 10),
    'decisiontreeclassifier__min_samples_leaf': range(3, 9, 2),
    'decisiontreeclassifier__min_samples_split': range(3, 40, 5),
    'decisiontreeclassifier__criterion':['gini', 'entropy']
    }

#building the grid search

grid_search = GridSearchCV(
                    grid_pipeline,
                    param_grid = grid_parameters,
                    cv=3, # the value for K in K-fold Cross Validation.
                    scoring='accuracy', # the performance metric to use,
                    verbose=1) # we want informative outputs during the training process

grid_search.fit(X_train, y_train)

In [ ]:
## you can visualize the best parameters
grid_search.best_params_

In [ ]:
## predicting
grid_test_predicted = grid_search.predict(X_test)

#measuring score
acc_test_grid = accuracy_score(y_test,grid_test_predicted)

acc_test_null, acc_test_num, acc_test_catnum, acc_test_grid

## Optimizing the full pipeline

You can also add grid parameters not only for the model, but also for the imputer, scaler, etc. In this case, we need to create another preprocessor.

In this case, it is better to use the function Pipeline(), instead of make_pipeline(). The only difference is that we have to be more explicit with the steps with Pipeline(), but it is better of optimizing other paramters in it.

In [ ]:
from sklearn.pipeline import Pipeline
## Create numerical pipeline with no parameters
numeric_pipe_2 = make_pipeline(
    SimpleImputer(),
    StandardScaler()
)

## Create categorical pipeline
categoric_pipe_2 = make_pipeline(
    SimpleImputer(strategy="constant", fill_value="missing"),
    OneHotEncoder(sparse_output=False, handle_unknown="ignore", drop="first")
)

## Fix: use numeric_pipe_2 and categoric_pipe_2 (not the old ones)
preprocessor_2 = make_column_transformer(
    (numeric_pipe_2, X_train_num.columns),
    (categoric_pipe_2, X_train_cat.columns),
)

## Use Pipeline with explicit names for easier parameter referencing
grid_pipeline_2 = Pipeline([
    ('preprocessor', preprocessor_2),
    ('classifier', DecisionTreeClassifier())
])

## Setting parameters grid with correct naming
grid_parameters_2 = {
    # Parameters for numeric pipeline (first pipeline in column transformer)
    'preprocessor__pipeline-1__simpleimputer__strategy': ["mean", "median"],
    'preprocessor__pipeline-1__standardscaler__with_mean': [True, False],
    'preprocessor__pipeline-1__standardscaler__with_std': [True, False],
    # Parameters for the decision tree classifier
    'classifier__max_depth': range(2, 10),
    'classifier__min_samples_leaf': range(3, 9, 2),
    'classifier__min_samples_split': range(3, 40, 5),
    'classifier__criterion': ['gini', 'entropy']
}

# Building the grid search
grid_search_2 = GridSearchCV(
    grid_pipeline_2,
    param_grid=grid_parameters_2,
    cv=3,
    scoring='accuracy',
    verbose=1,
    n_jobs=-1
)

grid_search_2.fit(X_train, y_train)

In [ ]:
## you can visualize the best parameters
grid_search_2.best_params_

In [ ]:
## predicting
grid_test_predicted_2 = grid_search_2.predict(X_test)

#measuring score
acc_test_grid_2 = accuracy_score(y_test,grid_test_predicted_2)

acc_test_null, acc_test_num, acc_test_catnum, acc_test_grid, acc_test_grid_2

## why a lower score?

## Optimizing with different models

What if we don't know which model is the best? RandomForest or Decisition Tree?

In [ ]:
from sklearn.ensemble import RandomForestClassifier

## Create numerical pipeline with no parameters
numeric_pipe_3 = make_pipeline(
    SimpleImputer(),
    StandardScaler()
)

## Create categorical pipeline
categoric_pipe_3 = make_pipeline(
    SimpleImputer(strategy="constant", fill_value="missing"),
    OneHotEncoder(sparse_output=False, handle_unknown="ignore", drop="first")
)

## Create preprocessor
preprocessor_3 = make_column_transformer(
    (numeric_pipe_3, X_train_num.columns),
    (categoric_pipe_3, X_train_cat.columns),
)

## Use Pipeline with explicit names
grid_pipeline_3 = Pipeline([
    ('preprocessor', preprocessor_3),
    ('model', DecisionTreeClassifier())  # placeholder model
])

## Setting parameters grid with both models
grid_parameters_3 = [
    # Decision Tree parameters
    {
        'preprocessor__pipeline-1__simpleimputer__strategy': ["mean", "median"],
        'preprocessor__pipeline-1__standardscaler__with_mean': [True, False],
        'preprocessor__pipeline-1__standardscaler__with_std': [True, False],
        'model': [DecisionTreeClassifier(random_state=42)],
        'model__max_depth': range(2, 10),
        'model__min_samples_leaf': range(3, 9, 2),
        'model__min_samples_split': range(3, 40, 5),
        'model__criterion': ['gini', 'entropy']
    },
    # Random Forest parameters
    {
        'preprocessor__pipeline-1__simpleimputer__strategy': ["mean", "median"],
        'preprocessor__pipeline-1__standardscaler__with_mean': [True, False],
        'preprocessor__pipeline-1__standardscaler__with_std': [True, False],
        'model': [RandomForestClassifier(random_state=42)],
        'model__n_estimators': [50, 100, 200],
        'model__max_depth': range(2, 10),
        'model__min_samples_leaf': range(3, 8),
        'model__criterion': ['gini', 'entropy']
    }
]

# Building the grid search
grid_search_3 = GridSearchCV(
    grid_pipeline_3,  # Fixed: was 'pipe4', now correct pipeline name
    param_grid=grid_parameters_3,
    cv=3,
    scoring='accuracy',
    verbose=1,
    n_jobs=-1
)

grid_search_3.fit(X_train, y_train)

In [ ]:
grid_search_3.best_params_

In [ ]:
## predicting
grid_test_predicted_3 = grid_search_3.predict(X_test)

#measuring score
acc_test_grid_3 = accuracy_score(y_test,grid_test_predicted_3)

acc_test_null, acc_test_num, acc_test_catnum, acc_test_grid, acc_test_grid_2, acc_test_grid_3

## why a lower score?

## Further Evaluations

Precision: Of all predicted survivors, how many actually survived?

Recall: Of all actual survivors, how many did we predict?

F1-Score: Harmonic mean of precision and recall

ROC-AUC: How well the model separates the two classes

In [ ]:
from sklearn.metrics import (
    precision_score, 
    recall_score, 
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
    roc_curve
)

# Get predictions for both models
y_pred_2 = grid_search_2.predict(X_test)
y_pred_3 = grid_search_3.predict(X_test)

# Get probability predictions for ROC curve
y_proba_2 = grid_search_2.predict_proba(X_test)[:, 1]
y_proba_3 = grid_search_3.predict_proba(X_test)[:, 1]

# Calculate metrics for grid_search_2
metrics_2 = {
    'Model': 'DecisionTree (grid_search_2)',
    'Accuracy': accuracy_score(y_test, y_pred_2),
    'Precision': precision_score(y_test, y_pred_2),
    'Recall': recall_score(y_test, y_pred_2),
    'F1-Score': f1_score(y_test, y_pred_2),
    'ROC-AUC': roc_auc_score(y_test, y_proba_2)
}

# Calculate metrics for grid_search_3
metrics_3 = {
    'Model': 'RandomForest (grid_search_3)',
    'Accuracy': accuracy_score(y_test, y_pred_3),
    'Precision': precision_score(y_test, y_pred_3),
    'Recall': recall_score(y_test, y_pred_3),
    'F1-Score': f1_score(y_test, y_pred_3),
    'ROC-AUC': roc_auc_score(y_test, y_proba_3)
}

# Create comparison dataframe
comparison_df = pd.DataFrame([metrics_2, metrics_3])
comparison_df

In [ ]:
# Confusion matrices visualization
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Confusion matrix for grid_search_2
cm_2 = confusion_matrix(y_test, y_pred_2)
axes[0].imshow(cm_2, cmap='Blues', alpha=0.7)
axes[0].set_title('DecisionTree (grid_search_2)\nConfusion Matrix')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')
axes[0].set_xticks([0, 1])
axes[0].set_yticks([0, 1])
axes[0].set_xticklabels(['Not Survived', 'Survived'])
axes[0].set_yticklabels(['Not Survived', 'Survived'])

for i in range(2):
    for j in range(2):
        axes[0].text(j, i, str(cm_2[i, j]), 
                    ha='center', va='center', fontsize=20, fontweight='bold')

# Confusion matrix for grid_search_3
cm_3 = confusion_matrix(y_test, y_pred_3)
axes[1].imshow(cm_3, cmap='Greens', alpha=0.7)
axes[1].set_title('RandomForest (grid_search_3)\nConfusion Matrix')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')
axes[1].set_xticks([0, 1])
axes[1].set_yticks([0, 1])
axes[1].set_xticklabels(['Not Survived', 'Survived'])
axes[1].set_yticklabels(['Not Survived', 'Survived'])

for i in range(2):
    for j in range(2):
        axes[1].text(j, i, str(cm_3[i, j]), 
                    ha='center', va='center', fontsize=20, fontweight='bold')

plt.tight_layout()
plt.gca()

In [ ]:
# ROC Curve comparison
fpr_2, tpr_2, _ = roc_curve(y_test, y_proba_2)
fpr_3, tpr_3, _ = roc_curve(y_test, y_proba_3)

plt.figure(figsize=(10, 6))
plt.plot(fpr_2, tpr_2, label=f'DecisionTree (AUC = {roc_auc_score(y_test, y_proba_2):.3f})', 
         linewidth=2, color='blue')
plt.plot(fpr_3, tpr_3, label=f'RandomForest (AUC = {roc_auc_score(y_test, y_proba_3):.3f})', 
         linewidth=2, color='green')
plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier', linewidth=1)
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curve Comparison', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(alpha=0.3)
plt.gca()

In [ ]:
# Metrics visualization
metrics_comparison = comparison_df.set_index('Model').T

plt.figure(figsize=(10, 6))
metrics_comparison.plot(kind='bar', width=0.8, color=['#1f77b4', '#2ca02c'])
plt.title('Model Performance Comparison', fontsize=14, fontweight='bold')
plt.ylabel('Score', fontsize=12)
plt.xlabel('Metric', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.ylim(0, 1)
plt.legend(title='Model', fontsize=10)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.gca()